# NonVee UIQ Info Job - Debug Notebook
Replicates `nonvee_uiq_info_job.py` step by step for validation on EMR Workspace.

**Job Parameters (from the actual run):**
- opco: `oh`
- batch_start_dttm_str: `2026-03-19 19:07:16`
- batch_end_dttm_str: `2026-03-24 10:45:21`
- batch_run_dttm_str: `2026-03-24 10:45:21`
- data_bucket: `aep-datalake-work-dev`

## Cell 1: Imports & Configuration

In [ ]:
import time
import uuid
import boto3

from typing import Dict
from textwrap import dedent
from functools import reduce
from collections import defaultdict
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException

import pyspark.sql.functions as F

print('Imports done.')

## Cell 2: Job Parameters & Constants

In [ ]:
# ============================================================
# Job parameters - same as the actual run
# ============================================================
job_name = 'nonvee-uiq-info-oh'
opco = 'oh'
data_bucket = 'aep-datalake-work-dev'
batch_start_dttm_str = '2026-03-19 19:07:16'
batch_end_dttm_str = '2026-03-24 10:45:21'
batch_run_dttm_str = '2026-03-24 10:45:21'

# ============================================================
# Constants
# ============================================================
data_prefix = 'raw/intervals/nonvee/uiq_info'
marker_file_name = 'index.txt'
marker_file_max_depth = 2
data_file_ext = '.xml'
data_file_min_size = 0

co_cd_ownr_map = {
    'oh': ['07', '10'],
    'im': ['04'],
    'pso': ['95'],
    'ap': ['01', '02', '06'],
    'swp': ['96']
}
timezone_map = {
    'oh': 'US/Eastern',
    'im': 'US/Eastern',
    'pso': 'US/Central',
    'ap': 'US/Eastern',
    'swp': 'US/Eastern'
}

xml_row_tag = 'MeterData'
xml_read_mode = 'FAILFAST'
xml_infer_schema = 'false'
xml_tag_ignore_namespace = 'true'
xml_data_timezone = 'America/New_York'
xml_tag_value_col_name = 'tag_value'

raw_xml_schema_str = """
`_MeterName` STRING,
`_UtilDeviceID` STRING,
`_MacID` STRING,
`IntervalReadData` ARRAY<
    STRUCT<
        `_NumberIntervals`: STRING,
        `_EndTime`: STRING,
        `_StartTime`: STRING,
        `_IntervalLength`: STRING,
        `Interval`: ARRAY<
            STRUCT<
                `_EndTime`: STRING,
                `_BlockSequenceNumber`: STRING,
                `_GatewayCollectedTime`: STRING,
                `_IntervalSequenceNumber`: STRING,
                `Reading`: ARRAY<
                    STRUCT<
                        `_Channel`: STRING,
                        `_RawValue`: STRING,
                        `_Value`: STRING,
                        `_UOM`: STRING,
                        `_BlockEndValue`: STRING
                    >
                >
            >
        >
    >
>
"""

print(f'opco: {opco}')
print(f'batch window: {batch_start_dttm_str} -> {batch_end_dttm_str}')

## Cell 3: Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName(job_name + '-debug') \
    .enableHiveSupport() \
    .getOrCreate()

print(f'Spark application-id: {spark.sparkContext.applicationId}')
print(f'Spark version: {spark.version}')

## Cell 4: Initialize Date/Time Variables

In [ ]:
current_dttm_ltz = datetime.now(timezone.utc).astimezone()
local_tz = current_dttm_ltz.tzinfo
batch_start_dttm_ntz = datetime.strptime(batch_start_dttm_str, '%Y-%m-%d %H:%M:%S')
batch_start_dttm_ltz = batch_start_dttm_ntz.replace(tzinfo=local_tz)
batch_end_dttm_ntz = datetime.strptime(batch_end_dttm_str, '%Y-%m-%d %H:%M:%S')
batch_end_dttm_ltz = batch_end_dttm_ntz.replace(tzinfo=local_tz)
batch_run_dttm_ntz = datetime.strptime(batch_run_dttm_str, '%Y-%m-%d %H:%M:%S')
batch_run_dttm_ltz = batch_run_dttm_ntz.replace(tzinfo=local_tz)
base_prefix = f'{data_prefix}/{opco}'
scratch_path = f'hdfs:///tmp/scratch/{str(uuid.uuid4())}'

print(f'current dttm: {current_dttm_ltz}')
print(f'batch start dttm: {batch_start_dttm_ltz}')
print(f'batch end dttm: {batch_end_dttm_ltz}')
print(f'batch run dttm: {batch_run_dttm_ltz}')
print(f'base prefix: {base_prefix}')
print(f'scratch path: {scratch_path}')

## Cell 5: S3 File Filtering (get_filtered_data_files_map)

In [ ]:
def get_filtered_data_files_map(
    s3_bucket, s3_prefix, marker_file_name,
    batch_start_dttm_ltz, batch_end_dttm_ltz,
    marker_file_max_depth, data_file_min_size, data_file_ext,
):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    base_prefix_len = len(s3_prefix.strip('/'))

    _time = time.time()
    print(f'===== listing objects in base prefix {s3_prefix} =====')
    i, j, k = 0, 0, 0
    all_files_map = defaultdict(list)
    filtered_dirs_prefix = set()

    for page in paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix):
        i += 1
        for obj in page.get('Contents', []):
            j += 1
            obj_key = obj['Key']
            obj_size = obj['Size']
            obj_last_modified = obj['LastModified']
            relative = obj_key[base_prefix_len:]
            relative_depth = relative.count('/')
            level1_dir = relative.strip('/').split('/')[0]
            level1_dir_prefix = f'{s3_prefix}/{level1_dir}'
            file_name = relative.split('/')[-1]

            if (
                obj_key.endswith(f'/{marker_file_name}')
                and batch_start_dttm_ltz <= obj_last_modified < batch_end_dttm_ltz
                and relative_depth <= marker_file_max_depth
            ):
                k += 1
                filtered_dirs_prefix.add(level1_dir_prefix)
            elif obj_size > data_file_min_size and data_file_ext in file_name:
                all_files_map[level1_dir_prefix].append(obj_key)

    print(f'===== found {j} objects in {i} pages =====')
    print(f'completed listing in {str(timedelta(seconds=time.time() - _time))}')

    filtered_files_map = {prefix: all_files_map[prefix] for prefix in filtered_dirs_prefix}
    print(f'===== selected {sum(len(f) for f in filtered_files_map.values())} files in {len(filtered_files_map)} dirs =====')
    return filtered_files_map


filtered_data_files_map = get_filtered_data_files_map(
    s3_bucket=data_bucket,
    s3_prefix=base_prefix,
    marker_file_name=marker_file_name,
    batch_start_dttm_ltz=batch_start_dttm_ltz,
    batch_end_dttm_ltz=batch_end_dttm_ltz,
    marker_file_max_depth=marker_file_max_depth,
    data_file_min_size=data_file_min_size,
    data_file_ext=data_file_ext,
)

# ============================================================
# DEBUG: Limit to 10 files total for faster testing
# ============================================================
MAX_FILES = 10
limited_files_map = {}
remaining = MAX_FILES
for prefix, files in filtered_data_files_map.items():
    if remaining <= 0:
        break
    limited_files_map[prefix] = files[:remaining]
    remaining -= len(limited_files_map[prefix])

filtered_data_files_map = limited_files_map
files_count = sum(len(f) for f in filtered_data_files_map.values())
print(f'Total files to process (limited to {MAX_FILES}): {files_count}')

## Cell 6: Read XML Files

In [ ]:
_time = time.time()
print('===== reading files =====')
files_count = 0
dfs = []
for k, files in filtered_data_files_map.items():
    files_count += len(files)
    files_str = ','.join([f's3://{data_bucket}/{f}' for f in files])
    df = spark.read.format('com.databricks.spark.xml') \
        .option('rowTag', xml_row_tag) \
        .option('mode', xml_read_mode) \
        .option('inferSchema', xml_infer_schema) \
        .option('valueTag', xml_tag_value_col_name) \
        .option('ignoreNamespace', xml_tag_ignore_namespace) \
        .option('timezone', xml_data_timezone) \
        .schema(raw_xml_schema_str) \
        .load(files_str)
    dfs.append(df)

print(f'===== read {files_count} files in {len(dfs)} dirs =====')
df_raw = reduce(lambda a, b: a.union(b), dfs)
print(f'completed reading in {str(timedelta(seconds=time.time() - _time))}')
df_raw.printSchema()

## Cell 7: Stage Raw Data

In [ ]:
_time = time.time()
print('===== staging raw data =====')
raw_staging_path = f'{scratch_path}/raw_stage/'
df_raw.coalesce(1000).write.mode('overwrite').parquet(raw_staging_path)
print(f'completed staging in {str(timedelta(seconds=time.time() - _time))}')

staged_raw_df = spark.read.parquet(raw_staging_path)
staged_raw_df_count = staged_raw_df.count()
print(f'Raw staged records: {staged_raw_df_count}')
staged_raw_df.printSchema()

## Cell 8: Read MACS Reference Table

In [ ]:
_time = time.time()
print('===== reading stg_nonvee.meter_premise_macs_ami =====')
macs_df = spark.table('stg_nonvee.meter_premise_macs_ami').filter(
    F.col('co_cd_ownr').isin(co_cd_ownr_map[opco])
).select(
    F.col('prem_nb'),
    F.col('bill_cnst'),
    F.col('acct_clas_cd'),
    F.col('acct_type_cd'),
    F.col('devc_cd'),
    F.col('pgm_id_nm'),
    F.concat(
        F.coalesce(F.col('tx_mand_data'), F.lit('')),
        F.coalesce(F.col('doe_nb'), F.lit('')),
        F.coalesce(F.col('serv_deliv_id'), F.lit('')),
    ).cast('string').alias('sd'),
    F.col('mfr_devc_ser_nbr'),
    F.col('mtr_inst_ts'),
    F.col('mtr_rmvl_ts'),
    F.col('acct_turn_on_dt'),
    F.col('acct_turn_off_dt'),
    F.col('bill_acct_nb'),
    F.col('mtr_pnt_nb'),
    F.col('tarf_pnt_nb'),
    F.lit(None).cast('string').alias('cmsg_mtr_mult_nb'),
    F.unix_timestamp('mtr_inst_ts', 'yyyy-MM-dd HH:mm:ss').alias('unix_mtr_inst_ts'),
    F.when(
        F.col('mtr_rmvl_ts') == '9999-01-01',
        F.unix_timestamp('mtr_rmvl_ts', 'yyyy-MM-dd')
    ).otherwise(
        F.unix_timestamp('mtr_rmvl_ts', 'yyyy-MM-dd HH:mm:ss')
    ).alias('unix_mtr_rmvl_ts'),
    F.unix_timestamp('acct_turn_on_dt', 'yyyy-MM-dd').alias('unix_acct_turn_on_dt'),
    F.unix_timestamp('acct_turn_off_dt', 'yyyy-MM-dd').alias('unix_acct_turn_off_dt'),
    F.col('serv_city_ad').alias('aep_city'),
    F.col('serv_zip_ad').alias('aep_zip'),
    F.col('state_cd').alias('aep_state'),
)
print(f'completed reading MACS in {str(timedelta(seconds=time.time() - _time))}')
print(f'MACS count: {macs_df.count()}')
macs_df.printSchema()

## Cell 9: Read UOM Mapping Table

In [ ]:
_time = time.time()
print('===== reading usage_nonvee.uom_mapping =====')
uom_mapping_df = spark.table('usage_nonvee.uom_mapping').filter(
    F.col('aep_opco') == opco
).select(
    F.col('aep_channel_id'),
    F.col('aep_derived_uom'),
    F.col('name_register'),
    F.col('aep_srvc_qlty_idntfr'),
    F.col('aep_raw_uom'),
    F.col('value_mltplr_flg'),
).distinct()
print(f'completed reading UOM mapping in {str(timedelta(seconds=time.time() - _time))}')
print(f'UOM mapping count: {uom_mapping_df.count()}')
uom_mapping_df.show(truncate=False)

## Cell 10: Step 1 - Rename Columns

In [ ]:
print('===== Step 1: Rename columns =====')
interval_data_files_src_df = staged_raw_df.withColumnRenamed(
    '_MeterName', 'metername'
).withColumnRenamed(
    '_UtilDeviceID', 'utildeviceid'
).withColumnRenamed(
    '_MacID', 'macid'
).withColumnRenamed(
    'IntervalReadData', 'interval_reading'
)
interval_data_files_src_df.printSchema()

## Cell 11: Step 2 - 1st EXPLODE (intervals)

In [ ]:
print('===== Step 2: 1st EXPLODE (intervals) =====')
interval_exploded_df = interval_data_files_src_df.withColumn(
    'exp_ird', F.explode('interval_reading')
).withColumn(
    'exp_interval', F.explode('exp_ird.Interval')
).select(
    F.col('metername').alias('MeterName'),
    F.col('utildeviceid').alias('UtilDeviceID'),
    F.col('macid').alias('MacID'),
    F.col('exp_ird._IntervalLength').alias('IntervalLength'),
    F.col('exp_ird._StartTime').alias('blockstarttime'),
    F.col('exp_ird._EndTime').alias('blockendtime'),
    F.col('exp_ird._NumberIntervals').alias('NumberIntervals'),
    F.col('exp_interval._EndTime').alias('int_endtime'),
    F.col('exp_interval._GatewayCollectedTime').alias('int_gatewaycollectedtime'),
    F.col('exp_interval._BlockSequenceNumber').alias('int_blocksequencenumber'),
    F.col('exp_interval._IntervalSequenceNumber').alias('int_intervalsequencenumber'),
    F.col('exp_interval.Reading').alias('int_reading'),
).filter(
    F.date_format(
        F.to_timestamp(F.substring('blockstarttime', 1, 10), 'yyyy-MM-dd'),
        'yyyyMMdd',
    ) >= F.lit('20150301')
)
interval_exploded_df.printSchema()

## Cell 12: Step 3 - 2nd EXPLODE (readings)

In [ ]:
print('===== Step 3: 2nd EXPLODE (readings) =====')
reading_exploded_df = interval_exploded_df.withColumn(
    'exp_reading', F.explode('int_reading')
).select(
    F.col('MeterName'),
    F.col('UtilDeviceID'),
    F.col('MacID'),
    F.col('IntervalLength'),
    F.col('blockstarttime'),
    F.col('blockendtime'),
    F.col('NumberIntervals'),
    F.from_unixtime(
        F.unix_timestamp(F.substring('int_endtime', 1, 19), "yyyy-MM-dd'T'HH:mm:ss")
        - (F.col('IntervalLength').cast('int') * 60),
        "yyyy-MM-dd'T'HH:mm:ss",
    ).alias('starttime'),
    F.from_unixtime(
        F.unix_timestamp(
            F.substring('int_endtime', 1, 19), "yyyy-MM-dd'T'HH:mm:ss"
        ), "yyyy-MM-dd'T'HH:mm:ss",
    ).alias('endtime'),
    F.substring('int_endtime', -6, 6).alias('interval_epoch'),
    F.col('int_gatewaycollectedtime'),
    F.col('int_blocksequencenumber'),
    F.col('int_intervalsequencenumber'),
    F.col('exp_reading._Channel').alias('channel'),
    F.col('exp_reading._RawValue').alias('rawvalue'),
    F.col('exp_reading._Value').alias('value'),
    F.col('exp_reading._UOM').alias('uom'),
).filter(
    F.isnotnull('channel')
    & F.isnotnull('rawvalue')
    & F.isnotnull('value')
    & F.isnotnull('uom')
)
reading_exploded_df.printSchema()
print('\n===== Sample reading_exploded_df (endtime + interval_epoch) =====')
reading_exploded_df.select('MeterName', 'endtime', 'interval_epoch', 'channel', 'value').show(5, truncate=False)

## Cell 13: Step 4 - LEFT JOIN with UOM mapping

In [ ]:
print('===== Step 4: LEFT JOIN with UOM mapping =====')
interval_data_files_src_vw_df = reading_exploded_df.alias('reading').join(
    uom_mapping_df.alias('um'),
    on=(F.col('reading.channel') == F.col('um.aep_channel_id')),
    how='left',
).select(
    F.trim(F.col('reading.MeterName')).alias('MeterName'),
    F.col('reading.UtilDeviceID'),
    F.col('reading.MacID'),
    F.col('reading.IntervalLength'),
    F.col('reading.blockstarttime'),
    F.col('reading.blockendtime'),
    F.col('reading.NumberIntervals'),
    F.concat(
        F.col('reading.starttime'),
        F.col('reading.interval_epoch'),
    ).cast('string').alias('starttime'),
    F.concat(
        F.col('reading.endtime'),
        F.col('reading.interval_epoch'),
    ).cast('string').alias('endtime'),
    F.col('reading.interval_epoch'),
    F.col('reading.int_gatewaycollectedtime'),
    F.col('reading.int_blocksequencenumber'),
    F.col('reading.int_intervalsequencenumber'),
    F.col('reading.channel'),
    F.col('reading.rawvalue'),
    F.col('reading.value'),
    F.lower(F.trim(F.col('reading.uom'))).alias('uom'),
    F.coalesce(F.col('um.aep_derived_uom'), F.lit('UNK')).cast('string').alias('aep_uom'),
    F.when(
        F.isnotnull('um.name_register'),
        F.expr("translate(um.name_register, 'mtr_ivl_len', reading.IntervalLength)"),
    ).otherwise(F.lit('UNK')).cast('string').alias('name_register'),
    F.coalesce(F.col('um.aep_srvc_qlty_idntfr'), F.lit('UNK')).cast('string').alias('aep_sqi'),
    F.col('um.value_mltplr_flg'),
    F.unix_timestamp(
        F.concat(F.col('reading.endtime'), F.col('reading.interval_epoch')),
        "yyyy-MM-dd'T'HH:mm:ssXXX"
    ).alias('unix_endtime'),
)
interval_data_files_src_vw_df.printSchema()

## Cell 14: Step 5 - LEFT JOIN with MACS

In [ ]:
print('===== Step 5: LEFT JOIN with MACS =====')
interval_data_files_stg_df = interval_data_files_src_vw_df.alias('src').join(
    macs_df.alias('macs'),
    on=(
        (F.col('src.MeterName') == F.col('macs.mfr_devc_ser_nbr'))
        & F.col('src.unix_endtime').between(
            F.col('macs.unix_mtr_inst_ts'), F.col('macs.unix_mtr_rmvl_ts'),
        )
        & F.col('src.unix_endtime').between(
            F.col('macs.unix_acct_turn_on_dt'), F.col('macs.unix_acct_turn_off_dt'),
        )
    ),
    how='left',
).select(
    F.lit(opco).cast('string').alias('aep_opco'),
    F.col('src.MeterName').alias('serialnumber'),
    F.col('src.UtilDeviceID').alias('utildeviceid'),
    F.col('src.MacID').alias('macid'),
    F.col('src.IntervalLength').alias('intervallength'),
    F.col('src.blockstarttime'),
    F.col('src.blockendtime'),
    F.col('src.NumberIntervals').alias('numberintervals'),
    F.col('src.starttime'),
    F.col('src.endtime'),
    F.col('src.interval_epoch'),
    F.col('src.int_gatewaycollectedtime'),
    F.col('src.int_blocksequencenumber'),
    F.col('src.int_intervalsequencenumber'),
    F.col('src.channel'),
    F.col('src.value').alias('aep_raw_value'),
    F.col('src.value'),
    F.col('src.uom').alias('aep_raw_uom'),
    F.upper(F.col('src.aep_uom')).alias('aep_derived_uom'),
    F.col('src.name_register'),
    F.col('src.aep_sqi').alias('aep_srvc_qlty_idntfr'),
    F.col('macs.prem_nb').alias('aep_premise_nb'),
    F.col('macs.bill_cnst'),
    F.col('macs.acct_clas_cd').alias('aep_acct_cls_cd'),
    F.col('macs.acct_type_cd').alias('aep_acct_type_cd'),
    F.col('macs.devc_cd').alias('aep_devicecode'),
    F.col('macs.pgm_id_nm').alias('aep_meter_program'),
    F.col('macs.sd').alias('aep_srvc_dlvry_id'),
    F.col('macs.mtr_pnt_nb').alias('aep_mtr_pnt_nb'),
    F.col('macs.tarf_pnt_nb').alias('aep_tarf_pnt_nb'),
    F.col('macs.cmsg_mtr_mult_nb').alias('aep_comp_mtr_mltplr'),
    F.col('macs.mtr_rmvl_ts').alias('aep_mtr_removal_ts'),
    F.col('macs.mtr_inst_ts').alias('aep_mtr_install_ts'),
    F.col('macs.aep_city'),
    F.substring(F.col('macs.aep_zip'), 1, 5).cast('string').alias('aep_zip'),
    F.col('macs.aep_state'),
    F.lit('info-insert').cast('string').alias('hdp_update_user'),
    F.col('src.value_mltplr_flg'),
    F.substring(F.trim(F.col('src.MeterName')), -2, 2).cast('string').alias('aep_meter_bucket'),
)
interval_data_files_stg_df.printSchema()

In [ ]:
print('===== Debug: unix_endtime + MACS Join Condition Columns (1 sample record) =====')
interval_data_files_src_vw_df.alias('src').join(
    macs_df.alias('macs'),
    on=(F.col('src.MeterName') == F.col('macs.mfr_devc_ser_nbr')),
    how='inner',
).select(
    F.col('src.MeterName'),
    F.col('macs.mfr_devc_ser_nbr'),
    F.col('src.endtime').alias('endtime_with_epoch'),
    F.col('src.unix_endtime'),
    F.from_unixtime(F.col('src.unix_endtime')).alias('unix_endtime_as_datetime'),
    F.col('macs.unix_mtr_inst_ts'),
    F.from_unixtime(F.col('macs.unix_mtr_inst_ts')).alias('mtr_inst_as_datetime'),
    F.col('macs.unix_mtr_rmvl_ts'),
    F.from_unixtime(F.col('macs.unix_mtr_rmvl_ts')).alias('mtr_rmvl_as_datetime'),
    F.col('macs.unix_acct_turn_on_dt'),
    F.from_unixtime(F.col('macs.unix_acct_turn_on_dt')).alias('acct_on_as_datetime'),
    F.col('macs.unix_acct_turn_off_dt'),
    F.from_unixtime(F.col('macs.unix_acct_turn_off_dt')).alias('acct_off_as_datetime'),
    F.when(
        F.col('src.unix_endtime').between(F.col('macs.unix_mtr_inst_ts'), F.col('macs.unix_mtr_rmvl_ts')),
        F.lit('YES')
    ).otherwise(F.lit('NO')).alias('in_mtr_window'),
    F.when(
        F.col('src.unix_endtime').between(F.col('macs.unix_acct_turn_on_dt'), F.col('macs.unix_acct_turn_off_dt')),
        F.lit('YES')
    ).otherwise(F.lit('NO')).alias('in_acct_window'),
).show(1, truncate=False, vertical=True)

## Cell 15: Check MACS Join Match Rate

In [ ]:
print('===== MACS Join Match Rate (FIXED) =====')
interval_data_files_stg_df.select(
    F.count('*').alias('total_rows'),
    F.sum(F.when(F.col('aep_premise_nb').isNull(), 1).otherwise(0)).alias('null_premise_count'),
    F.sum(F.when(F.col('aep_premise_nb').isNotNull(), 1).otherwise(0)).alias('matched_premise_count'),
    F.sum(F.when(F.col('bill_cnst').isNull(), 1).otherwise(0)).alias('null_bill_cnst_count'),
    F.sum(F.when(F.col('aep_city').isNull(), 1).otherwise(0)).alias('null_city_count'),
    F.sum(F.when(F.col('aep_meter_program').isNull(), 1).otherwise(0)).alias('null_meter_program_count'),
).show(truncate=False)

## Cell 17: Step 6 - STG_VW Transformations

In [ ]:
print('===== Step 6: STG_VW transformations =====')
interval_data_files_stg_vw_df = interval_data_files_stg_df.select(
    F.col('serialnumber'),
    F.lit('nonvee-hes').cast('string').alias('source'),
    F.col('aep_devicecode'),
    F.lit('N').cast('string').alias('isvirtual_meter'),
    F.col('interval_epoch').alias('timezoneoffset'),
    F.col('aep_premise_nb'),
    F.when(
        F.isnotnull('aep_premise_nb')
        & F.isnotnull('aep_tarf_pnt_nb')
        & F.isnotnull('aep_mtr_pnt_nb'),
        F.concat(
            F.col('aep_premise_nb'),
            F.lit('-'),
            F.col('aep_tarf_pnt_nb'),
            F.lit('-'),
            F.col('aep_mtr_pnt_nb'),
        ),
    ).otherwise(F.lit(None)).cast('string').alias('aep_service_point'),
    F.col('aep_srvc_dlvry_id'),
    F.col('name_register'),
    F.lit('N').cast('string').alias('isvirtual_register'),
    F.col('aep_derived_uom'),
    F.col('aep_raw_uom'),
    F.col('aep_srvc_qlty_idntfr'),
    F.col('channel').alias('aep_channel_id'),
    (F.col('intervallength').cast('int') * F.lit(60)).alias('aep_sec_per_intrvl'),
    F.lit(None).cast('string').alias('aep_meter_alias'),
    F.col('aep_meter_program'),
    F.lit('interval').cast('string').alias('aep_usage_type'),
    F.lit(timezone_map[opco]).alias('aep_timezone_cd'),
    F.col('endtime').alias('endtimeperiod'),
    F.col('starttime').alias('starttimeperiod'),
    F.when(
        F.col('value_mltplr_flg') == 'Y',
        F.col('value').cast('double') * F.col('bill_cnst').cast('double'),
    ).otherwise(F.col('value').cast('double')).alias('value'),
    F.col('aep_raw_value'),
    F.col('bill_cnst').alias('scalarfloat'),
    F.col('aep_acct_cls_cd'),
    F.col('aep_acct_type_cd'),
    F.col('aep_mtr_pnt_nb'),
    F.col('aep_tarf_pnt_nb'),
    F.col('aep_comp_mtr_mltplr').cast('double').alias('aep_comp_mtr_mltplr'),
    F.unix_timestamp(
        F.concat(F.col('endtime'), F.col('interval_epoch')),
        "yyyy-MM-dd'T'HH:mm:ssXXX",
    ).cast('string').alias('aep_endtime_utc'),
    F.col('aep_mtr_removal_ts'),
    F.col('aep_mtr_install_ts'),
    F.col('aep_city'),
    F.col('aep_zip'),
    F.col('aep_state'),
    F.current_timestamp().alias('hdp_insert_dttm'),
    F.current_timestamp().alias('hdp_update_dttm'),
    F.col('hdp_update_user'),
    F.substring(F.col('aep_premise_nb'), 1, 2).cast('string').alias('authority'),
    F.substring(F.col('starttime'), 1, 10).cast('string').alias('aep_usage_dt'),
    F.lit('new').cast('string').alias('data_type'),
    F.col('aep_opco'),
    F.col('aep_meter_bucket'),
)
interval_data_files_stg_vw_df.printSchema()

## Cell 18: Step 7 - XFRM Transformations

In [ ]:
print('===== Step 7: Deduplication + XFRM transformations =====')
xfrm_df = interval_data_files_stg_vw_df.select(
    F.col('serialnumber'),
    F.col('source'),
    F.col('aep_devicecode'),
    F.col('isvirtual_meter'),
    F.col('timezoneoffset'),
    F.col('aep_premise_nb'),
    F.col('aep_service_point'),
    F.col('aep_mtr_install_ts'),
    F.col('aep_mtr_removal_ts'),
    F.col('aep_srvc_dlvry_id'),
    F.col('aep_comp_mtr_mltplr'),
    F.col('name_register'),
    F.col('isvirtual_register'),
    F.lit(None).cast('string').alias('toutier'),
    F.lit(None).cast('string').alias('toutiername'),
    F.col('aep_srvc_qlty_idntfr'),
    F.col('aep_channel_id'),
    F.col('aep_raw_uom'),
    F.col('aep_sec_per_intrvl').cast('double'),
    F.col('aep_meter_alias'),
    F.col('aep_meter_program'),
    F.lit(None).cast('string').alias('aep_billable_ind'),
    F.col('aep_usage_type'),
    F.col('aep_timezone_cd'),
    F.col('endtimeperiod'),
    F.col('starttimeperiod'),
    F.col('value').cast('float'),
    F.col('aep_raw_value').cast('float'),
    F.col('scalarfloat').cast('float'),
    F.lit(None).cast('string').alias('aep_data_quality_cd'),
    F.lit(None).cast('string').alias('aep_data_validation'),
    F.col('aep_acct_cls_cd'),
    F.col('aep_acct_type_cd'),
    F.col('aep_mtr_pnt_nb'),
    F.col('aep_tarf_pnt_nb'),
    F.col('aep_endtime_utc'),
    F.col('aep_city'),
    F.col('aep_zip'),
    F.col('aep_state'),
    F.col('hdp_update_user'),
    F.lit(current_dttm_ltz).cast('timestamp').alias('hdp_insert_dttm'),
    F.lit(current_dttm_ltz).cast('timestamp').alias('hdp_update_dttm'),
    F.col('authority'),
    F.col('aep_derived_uom'),
    F.col('data_type'),
    F.col('aep_opco'),
    F.col('aep_usage_dt'),
    F.col('aep_meter_bucket'),
)
xfrm_df.printSchema()

## Cell 19: Stage XFRM Data

In [ ]:
_time = time.time()
print('===== staging xfrm_df =====')
xfrm_staging_path = f'{scratch_path}/xfrm_stage/'
xfrm_df.write.mode('overwrite').partitionBy('aep_usage_dt').parquet(xfrm_staging_path)
print(f'completed writing in {str(timedelta(seconds=time.time() - _time))}')

staged_xfrm_df = spark.read.parquet(xfrm_staging_path)
staged_xfrm_df_count = staged_xfrm_df.count()
print(f'XFRM staged records: {staged_xfrm_df_count}')

## Cell 20: Deduplication

In [ ]:
print('===== dedup staged_xfrm_df =====')
latest_upd_w = Window.partitionBy(
    'aep_usage_dt', 'aep_meter_bucket', 'serialnumber', 'aep_channel_id', 'aep_raw_uom', 'endtimeperiod',
).orderBy(F.desc('hdp_insert_dttm'))

deduped_xfrm_df = staged_xfrm_df.select(
    '*', F.row_number().over(latest_upd_w).alias('upd_rownum'),
).filter(F.col('upd_rownum') == 1).drop('upd_rownum')

_time = time.time()
print('===== staging deduped_xfrm_df =====')
deduped_xfrm_staging_path = f'{scratch_path}/dedup_xfrm_stage/'
deduped_xfrm_df.write.mode('overwrite').partitionBy('aep_usage_dt').parquet(deduped_xfrm_staging_path)
print(f'completed writing in {str(timedelta(seconds=time.time() - _time))}')

staged_deduped_xfrm_df = spark.read.parquet(deduped_xfrm_staging_path)
staged_deduped_xfrm_df_count = staged_deduped_xfrm_df.count()
print(f'Deduped XFRM records: {staged_deduped_xfrm_df_count}')
print(f'Records removed by dedup: {staged_xfrm_df_count - staged_deduped_xfrm_df_count}')

## Cell 21: Final Validation - Aggregate Counts

In [ ]:
print('===== Final Validation: Aggregate counts by aep_usage_dt =====')
staged_deduped_xfrm_df.groupBy('aep_usage_dt').agg(
    F.count('*').alias('total_records'),
    F.countDistinct('serialnumber').alias('unique_meters'),
    F.countDistinct('aep_channel_id').alias('unique_channels'),
    F.sum(F.when(F.col('aep_premise_nb').isNull(), 1).otherwise(0)).alias('null_premise_count'),
    F.sum(F.when(F.col('aep_premise_nb').isNotNull(), 1).otherwise(0)).alias('matched_premise_count'),
    F.sum(F.when(F.col('aep_derived_uom') == 'UNK', 1).otherwise(0)).alias('unknown_uom_count'),
    F.sum(F.when(F.col('value').isNull(), 1).otherwise(0)).alias('null_value_count'),
    F.sum(F.when(F.col('scalarfloat').isNull(), 1).otherwise(0)).alias('null_scalarfloat_count'),
    F.sum(F.when(F.col('aep_city').isNull(), 1).otherwise(0)).alias('null_city_count'),
    F.sum(F.when(F.col('authority').isNull(), 1).otherwise(0)).alias('null_authority_count'),
).orderBy('aep_usage_dt').show(truncate=False)

## Cell 22: Sample Records - Spot Check

In [ ]:
print('===== Sample records with premise populated =====')
staged_deduped_xfrm_df.filter(
    F.col('aep_premise_nb').isNotNull()
).select(
    'serialnumber', 'aep_usage_dt', 'aep_channel_id', 'aep_raw_uom',
    'endtimeperiod', 'starttimeperiod', 'value', 'aep_raw_value',
    'scalarfloat', 'aep_derived_uom', 'aep_premise_nb', 'aep_service_point',
    'aep_devicecode', 'aep_meter_program', 'aep_acct_cls_cd',
    'aep_city', 'aep_zip', 'aep_state', 'authority',
).show(10, truncate=False)

## Cell 23: Distinct Usage Dates (for reference)

In [ ]:
distinct_usage_dates = staged_deduped_xfrm_df.select('aep_usage_dt').distinct().collect()
distinct_usage_date_values = [r.aep_usage_dt for r in distinct_usage_dates]
usage_dates_str = ', '.join([f"'{d}'" for d in distinct_usage_date_values])
print(f'Distinct usage dates: {usage_dates_str}')

## Cell 25: Step 9 - MERGE into Consume Layer (DRY RUN)
This generates and prints the MERGE SQL but does NOT execute it.
Target: `usage_nonvee.reading_ivl_nonvee_{opco}`

In [ ]:
print('===== Step 9: MERGE into consume layer (DRY RUN) =====')

t_alias = 't'
s_alias = 's'
target_table_name = f'usage_nonvee.reading_ivl_nonvee_{opco}'
source_view_name = f'interval_data_xfrm_{opco}_vw'

sql_text = dedent(f"""
MERGE INTO {target_table_name} AS {t_alias}
USING {source_view_name} AS {s_alias}
   ON {t_alias}.aep_usage_dt in ({usage_dates_str})
  AND {t_alias}.aep_usage_dt = {s_alias}.aep_usage_dt
  AND {t_alias}.aep_meter_bucket = {s_alias}.aep_meter_bucket
  AND {t_alias}.serialnumber = {s_alias}.serialnumber
  AND {t_alias}.aep_channel_id = {s_alias}.aep_channel_id
  AND {t_alias}.aep_raw_uom = {s_alias}.aep_raw_uom
  AND {t_alias}.endtimeperiod = {s_alias}.endtimeperiod
 WHEN MATCHED THEN UPDATE SET
      {t_alias}.aep_srvc_qlty_idntfr = {s_alias}.aep_srvc_qlty_idntfr
    , {t_alias}.aep_channel_id = {s_alias}.aep_channel_id
    , {t_alias}.aep_sec_per_intrvl = {s_alias}.aep_sec_per_intrvl
    , {t_alias}.aep_meter_alias = {s_alias}.aep_meter_alias
    , {t_alias}.aep_meter_program = {s_alias}.aep_meter_program
    , {t_alias}.aep_usage_type = {s_alias}.aep_usage_type
    , {t_alias}.aep_timezone_cd = {s_alias}.aep_timezone_cd
    , {t_alias}.endtimeperiod = {s_alias}.endtimeperiod
    , {t_alias}.starttimeperiod = {s_alias}.starttimeperiod
    , {t_alias}.value = {s_alias}.value
    , {t_alias}.aep_raw_value = {s_alias}.aep_raw_value
    , {t_alias}.scalarfloat = {s_alias}.scalarfloat
    , {t_alias}.aep_acct_cls_cd = {s_alias}.aep_acct_cls_cd
    , {t_alias}.aep_acct_type_cd = {s_alias}.aep_acct_type_cd
    , {t_alias}.aep_mtr_pnt_nb = {s_alias}.aep_mtr_pnt_nb
    , {t_alias}.aep_comp_mtr_mltplr = {s_alias}.aep_comp_mtr_mltplr
    , {t_alias}.aep_endtime_utc = {s_alias}.aep_endtime_utc
    , {t_alias}.aep_mtr_removal_ts = {s_alias}.aep_mtr_removal_ts
    , {t_alias}.aep_mtr_install_ts = {s_alias}.aep_mtr_install_ts
    , {t_alias}.aep_city = {s_alias}.aep_city
    , {t_alias}.aep_zip = {s_alias}.aep_zip
    , {t_alias}.aep_state = {s_alias}.aep_state
    , {t_alias}.hdp_update_user = 'info-update'
    , {t_alias}.hdp_update_dttm = {s_alias}.hdp_update_dttm
    , {t_alias}.authority = {s_alias}.authority
 WHEN NOT MATCHED THEN INSERT (
      serialnumber, source, aep_devicecode, isvirtual_meter, timezoneoffset,
      aep_premise_nb, aep_service_point, aep_srvc_dlvry_id, name_register,
      isvirtual_register, toutier, toutiername, aep_derived_uom, aep_raw_uom,
      aep_srvc_qlty_idntfr, aep_channel_id, aep_sec_per_intrvl, aep_meter_alias,
      aep_meter_program, aep_billable_ind, aep_usage_type, aep_timezone_cd,
      endtimeperiod, starttimeperiod, value, aep_raw_value, scalarfloat,
      aep_data_quality_cd, aep_data_validation, aep_acct_cls_cd, aep_acct_type_cd,
      aep_mtr_pnt_nb, aep_tarf_pnt_nb, aep_comp_mtr_mltplr, aep_endtime_utc,
      aep_mtr_removal_ts, aep_mtr_install_ts, aep_city, aep_zip, aep_state,
      hdp_update_user, hdp_insert_dttm, hdp_update_dttm, authority,
      aep_opco, aep_usage_dt, aep_meter_bucket
    ) VALUES (
      {s_alias}.serialnumber, {s_alias}.source, {s_alias}.aep_devicecode,
      {s_alias}.isvirtual_meter, {s_alias}.timezoneoffset, {s_alias}.aep_premise_nb,
      {s_alias}.aep_service_point, {s_alias}.aep_srvc_dlvry_id, {s_alias}.name_register,
      {s_alias}.isvirtual_register, {s_alias}.toutier, {s_alias}.toutiername,
      {s_alias}.aep_derived_uom, {s_alias}.aep_raw_uom, {s_alias}.aep_srvc_qlty_idntfr,
      {s_alias}.aep_channel_id, {s_alias}.aep_sec_per_intrvl, {s_alias}.aep_meter_alias,
      {s_alias}.aep_meter_program, {s_alias}.aep_billable_ind, {s_alias}.aep_usage_type,
      {s_alias}.aep_timezone_cd, {s_alias}.endtimeperiod, {s_alias}.starttimeperiod,
      {s_alias}.value, {s_alias}.aep_raw_value, {s_alias}.scalarfloat,
      {s_alias}.aep_data_quality_cd, {s_alias}.aep_data_validation,
      {s_alias}.aep_acct_cls_cd, {s_alias}.aep_acct_type_cd, {s_alias}.aep_mtr_pnt_nb,
      {s_alias}.aep_tarf_pnt_nb, {s_alias}.aep_comp_mtr_mltplr, {s_alias}.aep_endtime_utc,
      {s_alias}.aep_mtr_removal_ts, {s_alias}.aep_mtr_install_ts, {s_alias}.aep_city,
      {s_alias}.aep_zip, {s_alias}.aep_state, 'info-insert',
      {s_alias}.hdp_insert_dttm, {s_alias}.hdp_update_dttm, {s_alias}.authority,
      {s_alias}.aep_opco, {s_alias}.aep_usage_dt, {s_alias}.aep_meter_bucket
    )
""")

print(sql_text)

# Register temp view (needed if you want to execute later)
staged_deduped_xfrm_df.createOrReplaceTempView(source_view_name)
print(f'Registered temp view: {source_view_name}')

# ============================================================
# UNCOMMENT TO EXECUTE THE MERGE:
# _time = time.time()
# spark.sql(sql_text)
# print(f'completed MERGE in {str(timedelta(seconds=time.time() - _time))}')
# ============================================================

## Cell 26: Step 10 - DELETE old + INSERT into Downstream Incremental Table (DRY RUN)
Target: `xfrm_interval.reading_ivl_nonvee_incr`

In [ ]:
print('===== Step 10: DELETE old + INSERT into downstream incr table (DRY RUN) =====')
target_incr_table = 'xfrm_interval.reading_ivl_nonvee_incr'

# Step 10a: DELETE old data (older than 8 days)
cutoff_date = (datetime.now() - timedelta(days=8)).strftime('%Y%m%d_%H%M%S')
delete_query = f"DELETE FROM {target_incr_table} WHERE aep_opco = '{opco}' AND run_date < '{cutoff_date}'"
print(f'DELETE query:\n{delete_query}')

# Step 10b: Build the incr DataFrame
reading_ivl_nonvee_incr_df = xfrm_df.filter(
    (F.col('data_type') == 'new') & (F.col('aep_opco') == opco)
).select(
    'serialnumber', 'source', 'aep_devicecode', 'isvirtual_meter', 'timezoneoffset',
    'aep_premise_nb', 'aep_service_point', 'aep_mtr_install_ts', 'aep_mtr_removal_ts',
    'aep_srvc_dlvry_id', 'aep_comp_mtr_mltplr', 'name_register', 'isvirtual_register',
    'toutier', 'toutiername', 'aep_srvc_qlty_idntfr', 'aep_channel_id', 'aep_raw_uom',
    'aep_sec_per_intrvl', 'aep_meter_alias', 'aep_meter_program', 'aep_billable_ind',
    'aep_usage_type', 'aep_timezone_cd', 'endtimeperiod', 'starttimeperiod',
    'value', 'aep_raw_value', 'scalarfloat', 'aep_data_quality_cd', 'aep_data_validation',
    'aep_acct_cls_cd', 'aep_acct_type_cd', 'aep_mtr_pnt_nb', 'aep_tarf_pnt_nb',
    'aep_endtime_utc', 'aep_city', 'aep_zip', 'aep_state',
    F.lit('info-insert').cast('string').alias('hdp_update_user'),
    F.col('hdp_insert_dttm').cast('timestamp').alias('hdp_insert_dttm'),
    F.col('hdp_update_dttm').cast('timestamp').alias('hdp_update_dttm'),
    'authority', 'aep_usage_dt', 'aep_derived_uom', 'aep_opco',
    F.lit(current_dttm_ltz.strftime('%Y%m%d_%H%M%S')).alias('run_date'),
)
print(f'\nIncr DataFrame schema:')
reading_ivl_nonvee_incr_df.printSchema()
print(f'Incr DataFrame count: {reading_ivl_nonvee_incr_df.count()}')

# ============================================================
# UNCOMMENT TO EXECUTE:
# spark.sql(delete_query)
# reading_ivl_nonvee_incr_df.writeTo(target_incr_table).append()
# print(f'Data appended to {target_incr_table}')
# ============================================================

## Cell 27: Compare 1 Record - Dev XFRM vs Prod XFRM Table vs Prod Consume Table
Pick a sample record from the dev output, then fetch the same record from:
1. Prod **xfrm table** (`stg_nonvee.interval_data_files_{opco}_xfrm`)
2. Prod **consume table** (`usage_nonvee.reading_ivl_nonvee_{opco}`)

to compare column by column.

In [ ]:
print('===== Pick 1 record: Dev XFRM vs Prod XFRM vs Prod Consume =====')

# Pick 1 record from dev (with premise populated for a meaningful comparison)
sample_row = staged_deduped_xfrm_df.filter(
    F.col('aep_premise_nb').isNotNull()
).limit(1).collect()[0]

sample_serial = sample_row['serialnumber']
sample_endtime = sample_row['endtimeperiod']
sample_channel = sample_row['aep_channel_id']
sample_usage_dt = sample_row['aep_usage_dt']
sample_uom = sample_row['aep_raw_uom']
sample_bucket = sample_row['aep_meter_bucket']

print(f'Sample key: serialnumber={sample_serial}, endtimeperiod={sample_endtime}, '
      f'aep_channel_id={sample_channel}, aep_raw_uom={sample_uom}, aep_usage_dt={sample_usage_dt}')

# --- Common filter for all 3 sources ---
def filter_by_key(df):
    return df.filter(
        (F.col('serialnumber') == sample_serial)
        & (F.col('endtimeperiod') == sample_endtime)
        & (F.col('aep_channel_id') == sample_channel)
        & (F.col('aep_raw_uom') == sample_uom)
    ).limit(1)

# --- 1. DEV record (from this notebook's xfrm output) ---
dev_record = filter_by_key(staged_deduped_xfrm_df)

# --- 2. PROD XFRM record ---
prod_xfrm_table = f'stg_nonvee.interval_data_files_{opco}_xfrm'
print(f'Reading from prod xfrm: {prod_xfrm_table}')
prod_xfrm_record = filter_by_key(
    spark.table(prod_xfrm_table).filter(F.col('aep_usage_dt') == sample_usage_dt)
)

# --- 3. PROD CONSUME record ---
prod_consume_table = f'usage_nonvee.reading_ivl_nonvee_{opco}'
print(f'Reading from prod consume: {prod_consume_table}')
prod_consume_record = filter_by_key(
    spark.table(prod_consume_table).filter(F.col('aep_usage_dt') == sample_usage_dt)
)

# --- Collect all ---
dev_row = dev_record.collect()
xfrm_row = prod_xfrm_record.collect()
consume_row = prod_consume_record.collect()

skip_cols = {'hdp_insert_dttm', 'hdp_update_dttm'}

# --- Print side by side ---
print(f'\n{"COLUMN":<30} {"DEV XFRM":<40} {"PROD XFRM":<40} {"PROD CONSUME":<40} {"MATCH"}')
print('-' * 200)

if dev_row:
    d = dev_row[0].asDict()
    x = xfrm_row[0].asDict() if xfrm_row else {}
    c = consume_row[0].asDict() if consume_row else {}

    for col_name in dev_record.columns:
        if col_name in skip_cols:
            continue
        dv = str(d.get(col_name, ''))
        xv = str(x.get(col_name, '')) if x else 'N/A'
        cv = str(c.get(col_name, '')) if c else 'N/A'

        if x and c:
            match = 'OK' if dv == xv == cv else '** DIFF **'
        elif x:
            match = 'OK' if dv == xv else '** DIFF **'
        elif c:
            match = 'OK' if dv == cv else '** DIFF **'
        else:
            match = 'NO PROD DATA'
        print(f'{col_name:<30} {dv:<40} {xv:<40} {cv:<40} {match}')

    if not xfrm_row:
        print(f'\n!! No matching record found in PROD XFRM table ({prod_xfrm_table})')
    if not consume_row:
        print(f'\n!! No matching record found in PROD CONSUME table ({prod_consume_table})')
else:
    print('!! No matching record found in DEV output')

## Cell 24: Cleanup Scratch Directory
Run this when you're done with the notebook.

In [ ]:
# Uncomment to cleanup scratch HDFS directory
# hadoop_conf = spark._jsc.hadoopConfiguration()
# fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
# path = spark._jvm.org.apache.hadoop.fs.Path(scratch_path)
# if fs.exists(path):
#     print(f'Deleting HDFS path: {path}')
#     fs.delete(path, True)
#     print('Deleted.')

print(f'Scratch path (delete manually if needed): {scratch_path}')